In [1]:
import pandas as pd
pd.set_option('display.expand_frame_repr', False)

f_1 = open('Tatoeba_uk_uk_1.txt', 'r', encoding='utf-8')
f_2 = open('Tatoeba_uk_uk_2.txt', 'r', encoding='utf-8')

col_1 = f_1.read().splitlines()
col_2 = f_2.read().splitlines()

print('number of statements in each file:',str(len(col_1))+',', len(col_2),'\n')

df = pd.DataFrame({'statement_1': col_1, 'statement_2': col_2})
df.to_csv('raw.csv', index=False)
print(df.info(),df.head(10), str(round(df.duplicated().sum()/(len(df)/100),2))+'% of all data are duplicates', sep='\n\n')

df = df.drop_duplicates()

# no data
df = df[(df['statement_1'].str.strip() != "") &
    (df['statement_2'].str.strip() != "")]
len(df)

# spaces
df['statement_1'] = df['statement_1'].str.strip().str.replace(r'\s+', ' ', regex=True)
df['statement_2'] = df['statement_2'].str.strip().str.replace(r'\s+', ' ', regex=True)

# apostrophe unification
df['statement_1'] = df['statement_1'].str.replace(r'[‘’`]', "'", regex=True)
df['statement_2'] = df['statement_2'].str.replace(r'[‘’`]', "'", regex=True)

print('duplicates are removed, rows with no info are removed, apostrophe is unified')

number of statements in each file: 842, 842 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 842 entries, 0 to 841
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   statement_1  842 non-null    object
 1   statement_2  842 non-null    object
dtypes: object(2)
memory usage: 13.3+ KB
None

                             statement_1                              statement_2
0                              Справді?                               Насправді? 
1                   Вона кинула палити.                      Вона кинула курити. 
2                             Йде сніг.                              Падає сніг. 
3                        У мене є мрія.                              Я маю мрію. 
4                  Будь ласка, не плач.                     Не плач, будь ласка. 
5  Майку подобається грати у баскетбол.   Майкові подобається грати в баскетбол. 
6                             Смачного!                          

In [2]:
words_n = df['statement_1'].str.split().str.len().sum() + df['statement_2'].str.split().str.len().sum()
symbols_n = df['statement_1'].str.len().sum() + df['statement_2'].str.len().sum()

print(words_n/len(df), 'words on average in each example\n',
      symbols_n/len(df), 'characters on average in each example\n')

df['symbols_n'] = df['statement_1'].str.len() + df['statement_2'].str.len()
df['words_n'] = df['statement_1'].str.split().str.len() + df['statement_2'].str.split().str.len()

median_symbols = df['symbols_n'].median()
median_words = df['words_n'].median()

print('median total symbols per row:', median_symbols,
      '\nmedian total words per row:', median_words,
      '\nexamples with <= 3 words:', (df['words_n']<=4).sum(),' -', round((df['words_n']<=3).sum()/(len(df)/100),2),'%')

7.9844124700239805 words on average in each example
 43.7589928057554 characters on average in each example

median total symbols per row: 41.0 
median total words per row: 8.0 
examples with <= 3 words: 95  - 2.76 %


In [3]:
df = df[['statement_1','statement_2']]
df.head()

,statement_1,statement_2
0,Справді?,Насправді?
1,Вона кинула палити.,Вона кинула курити.
2,Йде сніг.,Падає сніг.
3,У мене є мрія.,Я маю мрію.
4,"Будь ласка, не плач.","Не плач, будь ласка."


In [4]:
import numpy as np

df = df[['statement_1', 'statement_2']].copy()

labels = pd.Series([1] * len(df))

n_wrong = len(df)*2
wrong_examples = []
print(n_wrong,'\n')

np.random.seed(42)

statement_1_list = df['statement_1'].tolist()
statement_2_list = df['statement_2'].tolist()
existing_pairs = set(zip(statement_1_list, statement_2_list))

while len(wrong_examples) < n_wrong:
    s1 = np.random.choice(statement_1_list)
    s2 = np.random.choice(statement_2_list)
    
    # avoid creating an actual correct pair
    if (s1, s2) not in existing_pairs:
        wrong_examples.append((s1, s2))
        existing_pairs.add((s1, s2))  # prevent duplicates

df_wrong = pd.DataFrame(wrong_examples, columns=['statement_1', 'statement_2'])
df = pd.concat([df, df_wrong], ignore_index=True)

labels = pd.concat([labels, pd.Series([0] * n_wrong)], ignore_index=True)

shuffle_idx = np.random.permutation(len(df))
df = df.iloc[shuffle_idx].reset_index(drop=True)
labels = labels.iloc[shuffle_idx].reset_index(drop=True)

print(df.head(10))
labels.head(10)

1668 

               statement_1                    statement_2
0       Що? Я не чую тебе.             Що? Я тебе не чую.
1              Вони щезли.  Він не міг зробити подібного.
2  В тебе було повно часу.       В тебе було багато часу.
3               Я товстий.                     Я гладкий.
4    В мене багато друзів.            Вечірка скінчилася.
5  Я живу тут з 1990 року.           Це японський прапор.
6         Вже майже третя.        Я зі Сполучених Штатів.
7       У нас є білий кіт.                 Де ти мешкаєш?
8          Том має роботу.               У Тома є робота.
9        Засвітили світло.                 Що це значить?


0    1
1    0
2    1
3    1
4    0
5    0
6    0
7    0
8    1
9    0
dtype: int64

In [5]:
df.to_csv('processed.csv', index=False)
labels.to_csv('labels.csv', index=False)